In [0]:
import os
import tomlrt


In [0]:
%run "/Workspace/Users/x1088029@guest.telecomitalia.it/monitoring_tool_Utils"

In [0]:
# dbutils.widgets.text("root_path","/../../","root_dir")
dbutils.widgets.text("usecase name","bussiness_process_name")
 
# root_dir=dbutils.widgets.get("root_path")
use_case=dbutils.widgets.get("usecase name")


In [ ]:
%run "utils.py"

In [0]:

use_case=use_case.replace(" ","_").lower()
# this will be used to get data from dict
usecase_info=None
# checking if usecase is in configs dict
if use_case in use_case_with_freq.keys():
  usecase_info=use_case_with_freq[use_case]
  print(f" exists ✔️  \n",usecase_info)
else :
   raise ValueError("usecase not found in usecase_with_freq dictionary")


In [0]:
# fix path
dir_path = "/mnt"+ usecase_info["Raw_Data_Path"]

print(usecase_info["Raw_Data_Path"])

In [0]:
dbutils.fs.ls(dir_path)

In [0]:
# ================================================
# 1. LOAD ALL FILES
# ================================================
files_list = []

if isinstance(usecase_info["data_files"], dict):
    for subdirectory in usecase_info["data_files"]:
        try:
            files = dbutils.fs.ls(dir_path + "/" + subdirectory)
            files_list.extend(files)
        except Exception as e:
            print(f"⚠️ Cannot access {subdirectory}: {e}")
else:
    files_list.extend(dbutils.fs.ls(dir_path))

print(f"📂 Total files found: {len(files_list)}")


# ================================================
# 2. INIT TRACKERS
# ================================================
last_loaded_files = []
overall_latest_date = None
overall_latest_file = None


# ================================================
# 3. PROCESS LOGIC (FIXED)
# ================================================
if isinstance(usecase_info["data_files"], list):
    # ✅ CASE 1: LIST
    for files in usecase_info["data_files"]:
        print(f"\n📁 Processing pattern: {files}")

        matches = []
        regex = pattern_to_regex(files)
        
        for f in files_list:
            if re.search(regex, f.name):
                matches.append(f)

        if not matches:
            print(f"  ⚠️ No matching files for pattern: {files}")
            continue

        sorted_files = sorted(matches, key=lambda f: extract_date(f), reverse=True)
        latest = sorted_files[0]
        latest_date = extract_date(latest)

        if latest_date == datetime.min:
            print(f"  ⚠️ Invalid date for: {files}")
            continue

        # data_type = generic "root"
        last_loaded_files.append(latest.name)

        print(f"  ✔ Latest: {latest.name}")
        print(f"  📅 Date: {latest_date}")

        if overall_latest_date is None or latest_date > overall_latest_date:
            overall_latest_date = latest_date
            overall_latest_file = latest.name


else:
    # ✅ CASE 2: DICT
    for data_type, file_patterns in usecase_info["data_files"].items():
        print(f"\n📁 Processing: {data_type}")

        pattern_list = file_patterns if isinstance(file_patterns, list) else [file_patterns]

        for pattern in pattern_list:
            matches = []
            regex = pattern_to_regex(pattern)

            for f in files_list:
                if re.search(regex, f.name):
                    matches.append(f)

            if not matches:
                print(f"  ⚠️ No matching files for pattern: {pattern}")
                continue

            sorted_files = sorted(matches, key=lambda f: extract_date(f), reverse=True)
            latest = sorted_files[0]
            latest_date = extract_date(latest)

            if latest_date == datetime.min:
                print(f"  ⚠️ Invalid date for: {pattern}")
                continue

            last_loaded_files.append(latest.name)

            print(f"  ✔ Latest ({pattern}): {latest.name}")
            print(f"  📅 Date: {latest_date}")

            if overall_latest_date is None or latest_date > overall_latest_date:
                overall_latest_date = latest_date
                overall_latest_file = latest.name


# ================================================
# 4. SUMMARY
# ================================================
print("\n" + "=" * 60)
print("📊 SUMMARY")
print("=" * 60)

print(f"Total tracked: {len(last_loaded_files)}")

if overall_latest_file:
    print(f"\n🎯 Most recent file: {overall_latest_file} ({overall_latest_date})")

print("\n📋 Latest per pattern:")
for f in last_loaded_files:
    print(f)
    

In [0]:
usecase_info["deadline"]=get_deadline(
 datetime.now(),
    usecase_info["frequency"],
    0,
     2
)
 
# get_expected_date("weekly",0).strftime("%Y-%m-%d")

print(usecase_info["deadline"])


In [0]:
# checking file existence 
expected_files=[]
for _,files in usecase_info["data_files"].items():

   for i in files:
    # (print(f"{i}") )
    file_ex=expected_file=get_expected_file(
      file_formal=i,
      datamask=usecase_info["Date_mask"],
      frequency="D",
      target_weekday=0,
      lag_days=2)
    expected_files.append(file_ex)
    # print(expected_file)

print(expected_files)

In [0]:
# check if files matchs last loaded ones
missing_files=0
for file in expected_files:

  if file not in last_loaded_files:
    missing_files=missing_files+1
    print(f"file {file} not found")
    continue

print(missing_files)

In [0]:
if missing_files==0:
  print("No alarm defined ")
  dbutils.notebook.exit("OK")


In [0]:
# deadline=get_deadline(frequency="D",deadline_rule="daily",lag_days=2)
  #  sysdate: datetime,
  #   frequency: str,
  #   deadline_rule,
  #   lag_days:2
print(usecase_info)

_getting expected files on each usecase_

In [0]:
end_time = datetime.utcnow()
start_time = end_time - timedelta(days=1)
# GET LAST RUN BASED ON EACH USECASE
latest_run=retrieve_lastRun(usecase_info["pipeline_name"],logging=False,start_time=start_time,end_time=end_time)
# print(latest_run)
print("Pipeline Name:", latest_run.pipeline_name)
print("Run ID:", latest_run.run_id)
print("Status:", latest_run.status)
print("Start Time:", latest_run.run_start)

In [0]:
import pytz
"""
If sysdate is smaller than the next expected run, raise a level 1 alarm: the file is missing,
 but there are still days left to retrieve, request, or wait for it 
 """
next_expected_run=latest_run.run_start+timedelta(days=1)
today=datetime.now()
today = pytz.utc.localize(today)
if next_expected_run>today and latest_run.status =="Succeeded" :
  print(next_expected_run) 
  print("level 1 alarm: the file is missing, but there are still days left to retrieve, request, or wait for it ")


In [0]:
"""
If sysdate is between the next expected run and the use case deadline, raise a level 2 alarm: 
the file is missing and the use case is already waiting for delivery.
"""

if next_expected_run<today and today<usecase_info["deadline"]:
  print("alarm level 2 : the file is missing and the use case is already waiting for delivery")

In [0]:
"""
If sysdate is greater than the use case deadline, raise a level 3 alarm: 
the file is missing and the use case should already have been executed.
"""

if  today.date() > usecase_info["deadline"]:
  print("alarm level 3 : the file is missing and the use case should already have been executed.")